# Bài 5.5 · Object detection trên ảnh thật

**Câu hỏi của buổi này:** một detector trả về hàng trăm box kèm điểm. Làm sao biến chúng thành danh sách gọn, chấm đúng sai bằng IoU, và đọc ra AP?

Dữ liệu là **Penn-Fudan**: 170 ảnh chụp đường phố, mỗi người đi bộ có một box do người gán nhãn vẽ. Model là **Faster R-CNN** (torchvision) đã học trên COCO, dùng ngay, không train.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng số người, ảnh kèm box nhãn |
| 2. Detector trả về gì | bảng box và điểm, ảnh trước và sau khi lọc điểm |
| 3. IoU | ma trận IoU giữa box đoán và box nhãn |
| 4. NMS | cùng một ảnh khi tắt và bật NMS |
| 5. Precision, recall, AP | đường precision–recall, AP theo ngưỡng IoU, những box bị tính sai |
| 6. Đếm người | sai số đếm theo ngưỡng điểm |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 2 phút. Bài giảng: [ai-canuc80k · bài 5.5](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.5).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import zipfile, urllib.request
from pathlib import Path
import torch, torchvision
from torchvision.io import read_image, ImageReadMode
from torchvision.ops import box_iou, nms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from matplotlib.patches import Rectangle

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_IMG = 170
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Penn-Fudan lưu mỗi người thành một vùng mask; box nhãn là hình chữ nhật nhỏ nhất bao vùng ấy, dạng $(x_0, y_0, x_1, y_1)$ tính bằng pixel.

In [ ]:
root = Path('/tmp/pennfudan')
if not (root / 'PennFudanPed').exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve('https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip', root / 'data.zip')
    zipfile.ZipFile(root / 'data.zip').extractall(root)
img_paths = sorted((root / 'PennFudanPed/PNGImages').glob('*.png'))[:N_IMG]

def gt_boxes(img_path):
    m = read_image(str(root / 'PennFudanPed/PedMasks' / (img_path.stem + '_mask.png')))[0]
    return torch.stack([torch.tensor([xs.min(), ys.min(), xs.max(), ys.max()], dtype=torch.float32)
                        for ys, xs in (torch.nonzero(m == k, as_tuple=True) for k in m.unique()[1:])])
images = [read_image(str(p), ImageReadMode.RGB) for p in img_paths]
gts = [gt_boxes(p) for p in img_paths]
n_people = np.array([len(g) for g in gts])

table(pd.DataFrame([('số ảnh', len(images)), ('tổng số người có nhãn', int(n_people.sum())),
                    ('người mỗi ảnh', f'{n_people.min()}–{n_people.max()}, trung bình {n_people.mean():.2f}'),
                    ('cỡ ảnh', f'{min(i.shape[2] for i in images)}–{max(i.shape[2] for i in images)} pixel chiều ngang')],
                   columns=['', 'giá trị']), 'Penn-Fudan')
check('mỗi ảnh có ít nhất một người', n_people.min() >= 1)

def draw(ax, img, boxes, color, texts=None, lw=2):
    ax.imshow(img.permute(1, 2, 0)); ax.axis('off')
    for k, b in enumerate(boxes):
        x0, y0, x1, y1 = b.tolist()
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=color, lw=lw))
        if texts is not None:
            ax.text(x0, y0 - 4, texts[k], color='white', fontsize=8, bbox=dict(fc=color, ec='none', pad=1))

fig, axes = plt.subplots(1, 4, figsize=(15, 4.4))
for ax, i in zip(axes, [int(f * (len(images) - 1)) for f in (0, .15, .45, .75)]):
    draw(ax, images[i], gts[i], C['green']); ax.set_title(f'{len(gts[i])} người có nhãn', fontsize=10)
fig.suptitle('Box nhãn (xanh lá) do người vẽ', y=1.02)
plt.tight_layout(); plt.show()

## 2. Detector trả về gì

Với mỗi ảnh, model trả về ba mảng cùng độ dài: `boxes` (toạ độ), `labels` (một trong 91 mã lớp COCO), `scores` (độ tin từ 0 tới 1). Bản torchvision giữ mọi box có điểm từ 0.05 trở lên, đã qua NMS bên trong.

In [ ]:
W = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
with quiet():
    det = fasterrcnn_resnet50_fpn_v2(weights=W).to(device).eval()
cats, prep = W.meta['categories'], W.transforms()
PERSON = cats.index('person')

def detect(img):
    with torch.no_grad():
        out = det([prep(img).to(device)])[0]
    return {k: v.cpu() for k, v in out.items()}

i0 = int(np.argmax(n_people))
out0 = detect(images[i0])
top = pd.DataFrame({'lớp': [cats[l] for l in out0['labels'][:10]], 'điểm': out0['scores'][:10].numpy(),
                    'box (x0, y0, x1, y1)': [tuple(int(v) for v in b) for b in out0['boxes'][:10]]})
table(top, f'Mười box điểm cao nhất (ảnh có {n_people[i0]} người có nhãn; tổng {len(out0["scores"])} box)', fmt={'điểm': '{:.3f}'})

keep = (out0['labels'] == PERSON) & (out0['scores'] >= 0.5)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
draw(axes[0], images[i0], out0['boxes'], C['sand'], lw=1.2); axes[0].set_title(f'mọi box, mọi lớp, điểm ≥ 0.05: {len(out0["boxes"])} box', fontsize=11)
draw(axes[1], images[i0], gts[i0], C['green'])
draw(axes[1], images[i0], out0['boxes'][keep], C['clay'], [f'{s:.2f}' for s in out0['scores'][keep]])
axes[1].set_title(f'chỉ lớp person, điểm ≥ 0.5: {int(keep.sum())} box (đỏ cam) · nhãn: {n_people[i0]} (xanh lá)', fontsize=11)
plt.tight_layout(); plt.show()
check('ba mảng boxes, labels, scores cùng độ dài', len(out0['boxes']) == len(out0['labels']) == len(out0['scores']))

## 3. IoU: hai box chồng lên nhau bao nhiêu

$$\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

Bằng 1 khi trùng khít, bằng 0 khi không chạm nhau. Mỗi ô dưới đây là IoU giữa một box đoán (hàng) và một box nhãn (cột) của ảnh ở mục 2.

In [ ]:
P, G = out0['boxes'][keep], gts[i0]
iou = box_iou(P, G)
fig, ax = plt.subplots(figsize=(1.2 * len(G) + 2, 0.6 * len(P) + 1.6))
ax.imshow(iou, cmap='Blues', vmin=0, vmax=1); ax.grid(False)
for r in range(iou.shape[0]):
    for c in range(iou.shape[1]):
        ax.text(c, r, f'{iou[r, c]:.2f}', ha='center', va='center', fontsize=10, color='white' if iou[r, c] > .5 else '#333')
ax.set_xticks(range(len(G)), [f'nhãn {c + 1}' for c in range(len(G))]); ax.set_yticks(range(len(P)), [f'đoán {r + 1} ({s:.2f})' for r, s in enumerate(out0['scores'][keep])])
ax.set_title('IoU giữa box đoán và box nhãn')
plt.tight_layout(); plt.show()

a, b = P[0], G[int(iou[0].argmax())]
inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
check(f'tính tay IoU của đoán 1 với nhãn gần nhất: {inter / union:.4f}, khớp box_iou', abs(inter / union - iou[0].max()) < 1e-4)
note(f'Mỗi box nhãn có một box đoán với IoU cao nhất từ {iou.max(0).values.min():.2f} đến {iou.max(0).values.max():.2f}; '
     f'các ô còn lại gần 0 vì hai box ở hai người khác nhau.')

## 4. NMS: bỏ những box trùng nhau

Trước NMS, quanh mỗi người có hàng chục box gần giống nhau. NMS (non-maximum suppression) giữ box điểm cao nhất, xoá mọi box khác cùng lớp có IoU với nó vượt ngưỡng, rồi lặp lại. Ta tạm tắt NMS bên trong model để thấy điều đó.

In [ ]:
old = det.roi_heads.nms_thresh
det.roi_heads.nms_thresh = 1.01
raw = detect(images[i0])
det.roi_heads.nms_thresh = old
m = (raw['labels'] == PERSON) & (raw['scores'] >= 0.5)
rb, rs = raw['boxes'][m], raw['scores'][m]
kept = {t: nms(rb, rs, t) for t in [0.3, 0.5, 0.7, 0.9]}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
draw(axes[0], images[i0], rb, C['clay'], lw=1); axes[0].set_title(f'tắt NMS: {len(rb)} box person điểm ≥ 0.5', fontsize=11)
draw(axes[1], images[i0], rb[kept[0.5]], C['clay']); axes[1].set_title(f'NMS ngưỡng IoU 0.5: còn {len(kept[0.5])} box', fontsize=11)
plt.tight_layout(); plt.show()
table(pd.DataFrame([(t, len(k)) for t, k in kept.items()], columns=['ngưỡng IoU của NMS', 'số box còn lại']),
      f'Từ {len(rb)} box ban đầu (nhãn: {n_people[i0]} người)', fmt={'ngưỡng IoU của NMS': '{:.1f}'})
check('NMS giảm số box', len(kept[0.5]) < len(rb))
note(f'Ngưỡng thấp xoá mạnh tay: hai người đứng sát nhau có thể mất một. Ngưỡng cao giữ lại box trùng. Ảnh này: {len(kept[0.3])} box ở 0.3, {len(kept[0.9])} box ở 0.9.')

## 5. Precision, recall và AP trên cả bộ ảnh

Chạy detector trên mọi ảnh, giữ box lớp person. Để chấm, xếp mọi box đoán theo điểm giảm dần; mỗi box được **ghép** với box nhãn chưa ai nhận có IoU cao nhất, nếu IoU ≥ 0.5 thì là **TP**, không thì là **FP**. Box nhãn không ai nhận là **FN**. Mỗi box nhãn chỉ được ghép một lần.

Quét điểm từ cao xuống thấp tạo đường precision–recall; **AP** là diện tích dưới đường ấy.

In [ ]:
preds = []
for img in images:
    o = detect(img); k = o['labels'] == PERSON
    preds.append((o['boxes'][k], o['scores'][k]))

def match(thr):
    scores, tp = [], []
    for (pb, ps), g in zip(preds, gts):
        used = torch.zeros(len(g), dtype=torch.bool)
        ious = box_iou(pb, g) if len(pb) else torch.zeros(0, len(g))
        for r in ps.argsort(descending=True):
            cand = ious[r].clone(); cand[used] = -1
            j = int(cand.argmax()) if len(g) else -1
            hit = j >= 0 and cand[j] >= thr
            if hit: used[j] = True
            scores.append(float(ps[r])); tp.append(hit)
    order = np.argsort(-np.array(scores))
    return np.array(scores)[order], np.array(tp)[order]

def ap_curve(tp, n_gt):
    ctp = np.cumsum(tp); cfp = np.cumsum(~tp)
    rec, prec = ctp / n_gt, ctp / np.maximum(ctp + cfp, 1)
    mrec, mpre = np.r_[0, rec, 1], np.r_[0, prec, 0]
    for k in range(len(mpre) - 2, -1, -1): mpre[k] = max(mpre[k], mpre[k + 1])
    return float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:])), rec, prec

N_GT = int(n_people.sum())
s50, tp50 = match(0.5)
ap50, rec, prec = ap_curve(tp50, N_GT)
at05 = s50 >= 0.5
TP, FP = int(tp50[at05].sum()), int((~tp50[at05]).sum())
table(pd.DataFrame([('box đoán điểm ≥ 0.5', int(at05.sum())), ('TP', TP), ('FP', FP), ('FN', N_GT - TP),
                    ('precision', f'{TP / max(TP + FP, 1):.3f}'), ('recall', f'{TP / N_GT:.3f}')], columns=['ở điểm ≥ 0.5, IoU ≥ 0.5', 'giá trị']),
      f'{len(images)} ảnh, {N_GT} người có nhãn')

ths = np.round(np.arange(0.5, 0.96, 0.05), 2)
aps = [ap_curve(match(t)[1], N_GT)[0] for t in ths]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
axes[0].plot(rec, prec, color=C['blue'], lw=2)
k = int(np.argmin(np.abs(s50 - 0.5)))
axes[0].scatter([rec[k]], [prec[k]], color=C['clay'], zorder=3); axes[0].annotate('điểm ≥ 0.5', (rec[k], prec[k]), xytext=(-70, -25), textcoords='offset points', color=C['clay'])
axes[0].set(xlabel='recall', ylabel='precision', title=f'Precision–recall, IoU ≥ 0.5: AP = {ap50:.3f}', xlim=(0, 1.02), ylim=(0, 1.02))
axes[1].bar([f'{t:.2f}' for t in ths], aps, color=C['blue']); bar_labels(axes[1], '{:.2f}')
axes[1].set(xlabel='ngưỡng IoU để tính là đúng', ylabel='AP', title=f'AP giảm khi đòi box khít hơn; trung bình 0.50–0.95: {np.mean(aps):.3f}', ylim=(0, 1.05))
plt.tight_layout(); plt.show()
check(f'AP@0.5 = {ap50:.3f} vượt 0.7', ap50 > 0.7)
note(f'AP@0.5 là <b>{ap50:.3f}</b>; đòi IoU ≥ 0.75 thì còn {aps[list(ths).index(0.75)]:.3f}; trung bình mười ngưỡng 0.50–0.95 (cách COCO gọi là mAP) là <b>{np.mean(aps):.3f}</b>.')

Những box **điểm cao mà bị tính là FP**. Nhìn kỹ: box nào thật sự sai, box nào là người thật mà bộ nhãn bỏ sót (Penn-Fudan chỉ gán nhãn người đi bộ rõ, đứng gần)?

In [ ]:
fps = []
for idx, ((pb, ps), g) in enumerate(zip(preds, gts)):
    ious = box_iou(pb, g) if len(pb) else torch.zeros(0, len(g))
    for r in range(len(pb)):
        if ps[r] >= 0.8 and (len(g) == 0 or ious[r].max() < 0.5):
            fps.append((float(ps[r]), idx, pb[r]))
fps = sorted(fps, key=lambda t: -t[0])[:8]
if fps:
    fig, axes = plt.subplots(1, len(fps), figsize=(2 * len(fps), 3.4))
    for ax, (s, idx, b) in zip(np.atleast_1d(axes), fps):
        x0, y0, x1, y1 = [int(v) for v in b]
        ax.imshow(images[idx][:, y0:y1, x0:x1].permute(1, 2, 0)); ax.axis('off'); ax.set_title(f'điểm {s:.2f}', fontsize=10, color=C['clay'])
    fig.suptitle('Box điểm ≥ 0.8 không khớp box nhãn nào (IoU < 0.5)', y=1.04)
    plt.tight_layout(); plt.show()
n_fp8 = sum(1 for (pb, ps), g in zip(preds, gts) for r in range(len(pb)) if ps[r] >= 0.8 and box_iou(pb[r:r + 1], g).max() < 0.5)
note(f'Có <b>{n_fp8}</b> box như vậy; chúng kéo precision ở điểm ≥ 0.5 xuống {TP / max(TP + FP, 1):.2f}, trong khi AP@0.5 vẫn {ap50:.3f} vì box nhãn thường được điểm cao hơn. '
     f'Trước khi sửa model, hãy xem tận mắt FP: lỗi có thể nằm ở nhãn.')

## 6. Đếm người

Bài thi hay hỏi số đối tượng trong ảnh. Đếm số box person có điểm ≥ $t$, so với số người có nhãn; sai số đếm là trung bình $|\text{đếm} - \text{nhãn}|$ trên mọi ảnh.

In [ ]:
ts = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
cnt = {t: np.array([int((ps >= t).sum()) for _, ps in preds]) for t in ts}
err = pd.DataFrame([(t, np.abs(cnt[t] - n_people).mean(), (cnt[t] == n_people).mean(), (cnt[t] - n_people).mean()) for t in ts],
                   columns=['ngưỡng điểm', 'sai số đếm trung bình', 'tỉ lệ ảnh đếm đúng', 'đếm thừa trung bình'])
table(err, 'Đếm người theo ngưỡng điểm', fmt={'ngưỡng điểm': '{:.2f}', 'sai số đếm trung bình': '{:.3f}', 'tỉ lệ ảnh đếm đúng': '{:.3f}', 'đếm thừa trung bình': '{:+.3f}'},
      best='sai số đếm trung bình', lower_is_better=True)
tb = float(err.loc[err['sai số đếm trung bình'].idxmin(), 'ngưỡng điểm'])

fig, ax = plt.subplots(figsize=(6.5, 5))
jit = np.random.default_rng(0).uniform(-.15, .15, (2, len(n_people)))
ax.scatter(n_people + jit[0], cnt[tb] + jit[1], s=22, alpha=.6, color=C['blue'])
mx = max(n_people.max(), cnt[tb].max()) + 1
ax.plot([0, mx], [0, mx], color=C['gray'], ls='--', label='đếm đúng')
ax.set(xlabel='số người có nhãn', ylabel=f'số box điểm ≥ {tb}', title=f'Mỗi chấm một ảnh (ngưỡng {tb})'); ax.legend()
plt.tight_layout(); plt.show()
best_row = err.loc[err['ngưỡng điểm'] == tb].iloc[0]
note(f"Ngưỡng tốt nhất cho việc đếm là <b>{tb}</b>: sai trung bình {best_row['sai số đếm trung bình']:.2f} người mỗi ảnh, đếm đúng {best_row['tỉ lệ ảnh đếm đúng']:.0%} số ảnh. "
     f"Ngưỡng này khác ngưỡng cho recall cao: mục tiêu khác thì chọn ngưỡng khác, và chọn trên tập có nhãn."
     + (f" Ở mọi ngưỡng, detector đều đếm thừa (trung bình {best_row['đếm thừa trung bình']:+.2f}): một phần là những người thật không có box nhãn như ở mục 5." if (err['đếm thừa trung bình'] > 0).all() else ''))

## 7. Tóm tắt

In [ ]:
summary = [('số ảnh / số người có nhãn', f'{len(images)} / {N_GT}'),
           ('box trước và sau NMS (ảnh ở mục 4)', f'{len(rb)} → {len(kept[0.5])}'),
           ('precision / recall ở điểm ≥ 0.5, IoU ≥ 0.5', f'{TP / max(TP + FP, 1):.3f} / {TP / N_GT:.3f}'),
           ('AP@0.5', f'{ap50:.3f}'), ('AP trung bình IoU 0.50–0.95', f'{np.mean(aps):.3f}'),
           (f'sai số đếm ở ngưỡng {tb}', f"{best_row['sai số đếm trung bình']:.3f} người/ảnh")]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thay detector bằng `fasterrcnn_mobilenet_v3_large_fpn` (nhẹ hơn nhiều) rồi so AP và thời gian; đổi `nms_thresh` của model thành 0.3; chọn một ảnh FP ở mục 5 rồi tự vẽ thêm box nhãn để xem AP đổi bao nhiêu.